# 03_clean_gd

Builds `decide_catalog.decide_schema.barometer_gd` from:

1. **`250808_data_RGD_DECIDE.xlsx`** — historical GD export (wide → long, same as `GD.Rmd`).
   - `Samplenumber` = SHA-256 of Excel **`sample_id`** (1, 2, 3… within each `Dossier_ID`).

2. **`gd_labresults.csv`** (optional) — GD API pull uploaded to the volume (e.g. via Teams).
   - Already long format; **no `Dossier_ID` or `sample_id` in the API**.
   - **`Samplenumber` = SHA-256 of synthetic key**  
     `farmID|testDate|diagnosticTest|sampleType`  
     (documented API-derived sample key, not GD internal `sample_id`).
   - `Farm_ID` is taken from API `farmID` as-is (already encrypted by GD).

If both files exist, rows from both sources are combined before writing Delta.
Upload `gd_labresults.csv` to `/Volumes/decide_catalog/decide_schema/decide_volume/`.

In [ ]:
# Databricks notebook source

import hashlib
import os

import numpy as np
import pandas as pd
from pyspark.sql.types import DateType, DoubleType, StringType, StructField, StructType

CATALOG = "decide_catalog"
SCHEMA = "decide_schema"
VOLUME_PATH = "/Volumes/decide_catalog/decide_schema/decide_volume"
EXCEL_SOURCE = "250808_data_RGD_DECIDE.xlsx"
API_SOURCE = "gd_labresults.csv"
PATHOGEN_COLS = ["PM", "MH", "HS", "MB", "BRSV", "PI3", "BCV"]
OUTPUT_COLS = [
    "Lab_reference",
    "Country",
    "Breed",
    "Province",
    "Farm_ID",
    "Diagnostic_test",
    "Sample_type",
    "Samplenumber",
    "Date",
    "Date_month",
    "Date_week",
    "Pathogen",
    "Result",
]

spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.{SCHEMA}")


def should_run_pipeline():
    try:
        value = dbutils.jobs.taskValues.get(
            taskKey="00_check_source_changes", key="should_run", default="true"
        )
        return str(value).lower() == "true"
    except Exception:
        return True


def source_path(file_name):
    return f"{VOLUME_PATH}/{file_name}"


def volume_file_exists(file_name):
    try:
        dbutils.fs.ls(source_path(file_name))
        return True
    except Exception:
        return False


def sha256_hash(value):
    if pd.isna(value):
        return None
    return hashlib.sha256(str(value).encode("utf-8")).hexdigest()


def api_derived_samplenumber(row):
    """Synthetic sample ID for API rows (no dossier_id / sample_id in API)."""
    key = "|".join(
        [
            str(row.get("farmID", "")),
            str(row.get("testDate", "")),
            str(row.get("diagnosticTest", "")),
            str(row.get("sampleType", "")),
        ]
    )
    return sha256_hash(key)


def month_start(values):
    return pd.to_datetime(values, errors="coerce").dt.to_period("M").dt.start_time.dt.date


def week_start(values, week_start="sunday"):
    dates = pd.to_datetime(values, errors="coerce")
    freq = "W-SAT" if week_start == "sunday" else "W-SUN"
    return dates.dt.to_period(freq).dt.start_time.dt.date


def max_with_na(series):
    values = pd.to_numeric(series, errors="coerce")
    if values.notna().any():
        return values.max(skipna=True)
    return np.nan


def write_delta(pdf, table_name):
    pdf = pdf.reindex(columns=OUTPUT_COLS).copy()
    string_cols = [
        "Lab_reference",
        "Country",
        "Breed",
        "Province",
        "Farm_ID",
        "Diagnostic_test",
        "Sample_type",
        "Samplenumber",
        "Pathogen",
    ]
    for col in string_cols:
        pdf[col] = pdf[col].where(pd.notna(pdf[col]), None).astype(object)
    for col in ["Date", "Date_month", "Date_week"]:
        pdf[col] = pd.to_datetime(pdf[col], errors="coerce").dt.date
    pdf["Result"] = pd.to_numeric(pdf["Result"], errors="coerce")

    schema = StructType(
        [
            StructField("Lab_reference", StringType(), True),
            StructField("Country", StringType(), True),
            StructField("Breed", StringType(), True),
            StructField("Province", StringType(), True),
            StructField("Farm_ID", StringType(), True),
            StructField("Diagnostic_test", StringType(), True),
            StructField("Sample_type", StringType(), True),
            StructField("Samplenumber", StringType(), True),
            StructField("Date", DateType(), True),
            StructField("Date_month", DateType(), True),
            StructField("Date_week", DateType(), True),
            StructField("Pathogen", StringType(), True),
            StructField("Result", DoubleType(), True),
        ]
    )
    sdf = spark.createDataFrame(pdf, schema=schema)
    full_name = f"{CATALOG}.{SCHEMA}.{table_name}"
    (
        sdf.write.format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .saveAsTable(full_name)
    )
    print(f"Wrote {sdf.count()} rows to {full_name}")


def clean_gd_excel():
    raw = pd.read_excel(source_path(EXCEL_SOURCE), engine="openpyxl")
    df = raw.rename(
        columns={
            "Dossier_ID": "Filenumber",
            "sample_id": "Samplenumber",
            "farm_ID": "Farm_ID",
            "project": "Project",
            "date": "Date",
        }
    )
    df["Country"] = "The Netherlands"
    df["Lab_reference"] = "2"
    df["Sample_type"] = np.select(
        [
            df["reason_of_sampling"].eq("Autopsy"),
            df["sample"].eq("BAL"),
            df["sample"].eq("SWABS"),
            df["sample"].eq("OTHER"),
        ],
        ["Autopsy", "BAL", "Swab", "Unknown"],
        default="Missing",
    )
    df["Diagnostic_test"] = df["test"].map({"PCR": "PCR", "Kweek": "Culture"}).fillna("Missing")
    df["Breed"] = (
        df["breed"]
        .map(
            {
                "beef": "Beef",
                "dairy": "Dairy",
                "mixed": "Mixed",
                "veal": "Veal",
                "other": "Unknown",
                "rearing": "Unknown",
                "unknown": "Unknown",
            }
        )
        .fillna("Unknown")
    )
    df["Province"] = (
        df["provincie"]
        .map(
            {
                "DR": "Drenthe",
                "FL": "Flevoland",
                "FR": "Friesland",
                "GL": "Gelderland",
                "GR": "Groningen",
                "LB": "Limburg",
                "NB": "North Brabant",
                "NH": "North Holland",
                "OV": "Overijssel",
                "UT": "Utrecht",
                "ZH": "South Holland",
                "ZL": "Zeeland",
            }
        )
        .fillna("Missing")
    )
    df = df[
        [
            "Filenumber",
            "Diagnostic_test",
            "Samplenumber",
            "Country",
            "Lab_reference",
            "Sample_type",
            "Breed",
            *PATHOGEN_COLS,
            "Date",
            "Province",
            "Project",
            "Farm_ID",
        ]
    ].drop_duplicates()
    for col in ["Filenumber", "Samplenumber", "Farm_ID"]:
        df[col] = df[col].apply(sha256_hash)
    df["Date"] = pd.to_datetime(df["Date"], errors="coerce", dayfirst=True)
    df["Date_month"] = month_start(df["Date"])
    df["Date_week"] = week_start(df["Date"], week_start="sunday")
    df = df[df["Project"].isin(["monitoring", "no project"])]
    group_cols = [
        "Lab_reference",
        "Country",
        "Breed",
        "Province",
        "Farm_ID",
        "Diagnostic_test",
        "Sample_type",
        "Samplenumber",
        "Date_month",
        "Date_week",
        "Date",
    ]
    grouped = df.groupby(group_cols, dropna=False)[PATHOGEN_COLS].agg(max_with_na).reset_index()
    long_df = grouped.melt(
        id_vars=group_cols,
        value_vars=PATHOGEN_COLS,
        var_name="Pathogen",
        value_name="Result",
    )
    long_df["data_source"] = "excel"
    print(f"Excel source: {len(long_df)} long rows from {EXCEL_SOURCE}")
    return long_df[OUTPUT_COLS + ["data_source"]]


def clean_gd_api():
    api = pd.read_csv(source_path(API_SOURCE))
    api["Samplenumber"] = api.apply(api_derived_samplenumber, axis=1)
    api = api.rename(
        columns={
            "labReference": "Lab_reference",
            "country": "Country",
            "breed": "Breed",
            "testDate": "Date",
            "province": "Province",
            "farmID": "Farm_ID",
            "diagnosticTest": "Diagnostic_test",
            "sampleType": "Sample_type",
            "pathogen": "Pathogen",
            "result": "Result",
        }
    )
    api["Lab_reference"] = api["Lab_reference"].astype(str)
    api["Date"] = pd.to_datetime(api["Date"], errors="coerce", utc=True).dt.tz_convert(None)
    api["Date_month"] = month_start(api["Date"])
    api["Date_week"] = week_start(api["Date"], week_start="sunday")
    api["Breed"] = api["Breed"].fillna("Unknown")
    api["Sample_type"] = api["Sample_type"].fillna("Missing")
    api["Diagnostic_test"] = api["Diagnostic_test"].replace({"CULTURE": "Culture"}).fillna("Missing")
    api["Pathogen"] = api["Pathogen"].astype(str).str.upper()
    api = api[api["Pathogen"].isin(PATHOGEN_COLS)]
    group_cols = [
        "Lab_reference",
        "Country",
        "Breed",
        "Province",
        "Farm_ID",
        "Diagnostic_test",
        "Sample_type",
        "Samplenumber",
        "Date_month",
        "Date_week",
        "Date",
        "Pathogen",
    ]
    api = (
        api.groupby(group_cols, dropna=False)["Result"]
        .agg(max_with_na)
        .reset_index()
    )
    api["data_source"] = "api"
    print(
        f"API source: {len(api)} long rows from {API_SOURCE} "
        "(Samplenumber = SHA-256 of farmID|testDate|diagnosticTest|sampleType)"
    )
    return api[OUTPUT_COLS + ["data_source"]]


if not should_run_pipeline():
    dbutils.notebook.exit("No source changes detected; skipping GD")

parts = []
if volume_file_exists(EXCEL_SOURCE):
    parts.append(clean_gd_excel())
else:
    raise FileNotFoundError(f"Required source missing in volume: {EXCEL_SOURCE}")

if volume_file_exists(API_SOURCE):
    parts.append(clean_gd_api())
else:
    print(f"Optional source not found (skipped): {API_SOURCE}")

barometer = pd.concat(parts, ignore_index=True)
source_mix = barometer["data_source"].value_counts().to_dict()
print(f"Combined rows by source: {source_mix}")
barometer = barometer.drop(columns=["data_source"])
write_delta(barometer, "barometer_gd")
